In [1]:
import os

# see what got uploaded
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        print(os.path.join(root, f))

/kaggle/input/datasets/arkaloveskaggle/prescription-preprocessed/preprocessed/Ether-5_png_jpg.rf.Bw5VPDuvZ2CoFbLZ4PVk.jpg
/kaggle/input/datasets/arkaloveskaggle/prescription-preprocessed/preprocessed/Antigen-1_png_jpg.rf.Gq7SALfR27kSf5hsZoeS.jpg
/kaggle/input/datasets/arkaloveskaggle/prescription-preprocessed/preprocessed/Probenicid-1_png_jpg.rf.oJO0YFJfLSx93J4SzHX6.jpg
/kaggle/input/datasets/arkaloveskaggle/prescription-preprocessed/preprocessed/Triethanolamine-5_png_jpg.rf.mPu84mP9W0bzuztURFan.jpg
/kaggle/input/datasets/arkaloveskaggle/prescription-preprocessed/preprocessed/340688370_1368138737367286_142861896250244192_n_jpg.rf.BRAt2GQkBgfJxgroMwMD.jpg
/kaggle/input/datasets/arkaloveskaggle/prescription-preprocessed/preprocessed/Lobelia-Alcolide-2_png_jpg.rf.fi9KQKmIFzP5LPmcvKYe.jpg
/kaggle/input/datasets/arkaloveskaggle/prescription-preprocessed/preprocessed/1151_jpg.rf.nqo5vBYrXdLr3JMnDdgx.jpg
/kaggle/input/datasets/arkaloveskaggle/prescription-preprocessed/preprocessed/Vinblastine

In [2]:
import os

dataset_path = '/kaggle/input/datasets/arkaloveskaggle/prescription-preprocessed/preprocessed'
images = os.listdir(dataset_path)
print(f"Total images: {len(images)}")
print("Sample:", images[:3])

Total images: 3023
Sample: ['Ether-5_png_jpg.rf.Bw5VPDuvZ2CoFbLZ4PVk.jpg', 'Antigen-1_png_jpg.rf.Gq7SALfR27kSf5hsZoeS.jpg', 'Probenicid-1_png_jpg.rf.oJO0YFJfLSx93J4SzHX6.jpg']


In [3]:
# install libraries
!pip install transformers jiwer -q

import os
import torch
import shutil
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using: {device}")
print(f"GPU: {torch.cuda.get_device_name(0)}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 12.4 MB/s eta 0:00:0000:010:01
Using: cuda
GPU: Tesla T4


In [6]:
# test the label fix
test_files = [
    '-a-Benzyl-Penicillin-2_png_jpg.rf.xxx.jpg',
    'Ethosoximide-4_png_jpg.rf.xxx.jpg',
    'Benzyl-morphine-1_png_jpg.rf.xxx.jpg'
]

def get_actual_label(filename):
    name = filename.lstrip('-')
    parts = name.split('-')
    clean_parts = []
    for part in parts:
        if part.replace('_','').replace('png','').replace('jpg','').isdigit():
            break
        if 'png' in part.lower() or 'jpg' in part.lower():
            break
        clean_parts.append(part)
    return ' '.join(clean_parts).strip().lower()

for f in test_files:
    print(f"{f[:45]} → '{get_actual_label(f)}'")

-a-Benzyl-Penicillin-2_png_jpg.rf.xxx.jpg → 'a benzyl penicillin'
Ethosoximide-4_png_jpg.rf.xxx.jpg → 'ethosoximide'
Benzyl-morphine-1_png_jpg.rf.xxx.jpg → 'benzyl morphine'


In [7]:
processor = TrOCRProcessor.from_pretrained('microsoft/trocr-base-handwritten')
model = VisionEncoderDecoderModel.from_pretrained('microsoft/trocr-base-handwritten')

model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.decoder_start_token_id = processor.tokenizer.cls_token_id

model = model.to(device)
print("TrOCR loaded!")

preprocessor_config.json:   0%|          | 0.00/224 [00:00<?, ?B/s]

The image processor of type `ViTImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/1.33G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/478 [00:00<?, ?it/s]

VisionEncoderDecoderModel LOAD REPORT from: microsoft/trocr-base-handwritten
Key                         | Status  | 
----------------------------+---------+-
encoder.pooler.dense.bias   | MISSING | 
encoder.pooler.dense.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


generation_config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

TrOCR loaded!


In [10]:
class PrescriptionDataset(Dataset):
    def __init__(self, image_folder, processor):
        self.image_folder = image_folder
        self.processor = processor
        self.images = [f for f in os.listdir(image_folder) 
                      if f.endswith(('.jpg', '.jpeg', '.png'))]
    
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        filename = self.images[idx]
        medicine_name = filename.split('-')[0].replace('_', ' ').strip()
        
        image_path = os.path.join(self.image_folder, filename)
        image = Image.open(image_path).convert('RGB')
        
        pixel_values = self.processor(
            image, return_tensors='pt').pixel_values.squeeze()
        labels = self.processor.tokenizer(
            medicine_name,
            padding='max_length',
            max_length=64,
            truncation=True,
            return_tensors='pt'
        ).input_ids.squeeze()
        
        labels[labels == self.processor.tokenizer.pad_token_id] = -100
        return pixel_values, labels

dataset = PrescriptionDataset('/kaggle/input/datasets/arkaloveskaggle/prescription-preprocessed/preprocessed', processor)
dataloader = DataLoader(dataset, batch_size=8, shuffle=True)
print(f"Dataset ready: {len(dataset)} images")

Dataset ready: 3023 images


In [11]:
dataset = PrescriptionDataset(dataset_path, processor)
dataloader = DataLoader(dataset, batch_size=8, shuffle=True)
print(f"Dataset ready: {len(dataset)} images")

Dataset ready: 3023 images


In [ ]:
num_epochs = 10
optimizer = AdamW(model.parameters(), lr=2e-5)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=100,
    num_training_steps=len(dataloader) * num_epochs
)

model.train()

for epoch in range(num_epochs):
    total_loss = 0
    for batch_idx, (pixel_values, labels) in enumerate(dataloader):
        pixel_values = pixel_values.to(device)
        labels = labels.to(device)
        
        outputs = model(pixel_values=pixel_values, labels=labels)
        loss = outputs.loss
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()
        
        total_loss += loss.item()
        
        if batch_idx % 50 == 0:
            print(f"Epoch {epoch+1}/{num_epochs} | Batch {batch_idx}/{len(dataloader)} | Loss: {loss.item():.4f}")
    
    avg_loss = total_loss / len(dataloader)
    print(f"\nEpoch {epoch+1} complete | Avg Loss: {avg_loss:.4f}\n")
    
    # save checkpoint after every epoch
    torch.save({
        'epoch': epoch,
        'model_state': model.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'loss': avg_loss
    }, '/kaggle/working/checkpoint.pth')
    print(f"Checkpoint saved!")

# save final model
model.save_pretrained('/kaggle/working/prescription_model')
processor.save_pretrained('/kaggle/working/prescription_model')
print("Training complete! Model saved!")

Epoch 1/10 | Batch 0/378 | Loss: 10.8703
Epoch 1/10 | Batch 50/378 | Loss: 4.2218
Epoch 1/10 | Batch 100/378 | Loss: 2.9710
Epoch 1/10 | Batch 150/378 | Loss: 2.0333
Epoch 1/10 | Batch 200/378 | Loss: 3.0626
Epoch 1/10 | Batch 250/378 | Loss: 3.7083
Epoch 1/10 | Batch 300/378 | Loss: 3.0980
Epoch 1/10 | Batch 350/378 | Loss: 2.2637

Epoch 1 complete | Avg Loss: 3.1933

Checkpoint saved!
Epoch 2/10 | Batch 0/378 | Loss: 2.6952
Epoch 2/10 | Batch 50/378 | Loss: 3.1562
Epoch 2/10 | Batch 100/378 | Loss: 2.3567
Epoch 2/10 | Batch 150/378 | Loss: 0.0348
Epoch 2/10 | Batch 200/378 | Loss: 2.1410
Epoch 2/10 | Batch 250/378 | Loss: 2.9119
Epoch 2/10 | Batch 300/378 | Loss: 3.2405
Epoch 2/10 | Batch 350/378 | Loss: 2.4257


In [ ]:
# load checkpoint and continue from epoch 7




checkpoint = torch.load('/kaggle/working/checkpoint.pth')
model.load_state_dict(checkpoint['model_state'])
optimizer.load_state_dict(checkpoint['optimizer_state'])
start_epoch = checkpoint['epoch'] + 1
print(f"Resuming from epoch {start_epoch}")

# continue training
for epoch in range(start_epoch, 10):
    total_loss = 0
    for batch_idx, (pixel_values, labels) in enumerate(dataloader):
        pixel_values = pixel_values.to(device)
        labels = labels.to(device)
        
        outputs = model(pixel_values=pixel_values, labels=labels)
        loss = outputs.loss
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()
        
        total_loss += loss.item()
        
        if batch_idx % 50 == 0:
            print(f"Epoch {epoch+1}/10 | Batch {batch_idx}/{len(dataloader)} | Loss: {loss.item():.4f}")
    
    avg_loss = total_loss / len(dataloader)
    print(f"\nEpoch {epoch+1} complete | Avg Loss: {avg_loss:.4f}\n")
    
    torch.save({
        'epoch': epoch,
        'model_state': model.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'loss': avg_loss
    }, '/kaggle/working/checkpoint.pth')
    print("Checkpoint saved!")

In [ ]:
from PIL import Image
from jiwer import cer, wer
from tqdm import tqdm
import torch

model.eval()

def predict(image_path):
    image = Image.open(image_path).convert('RGB')
    pixel_values = processor(image, return_tensors='pt').pixel_values.to(device)
    
    with torch.no_grad():
        generated_ids = model.generate(pixel_values)
    
    return processor.batch_decode(generated_ids, skip_special_tokens=True)[0].lower().strip()

def get_actual_label(filename):
    name = filename.split('-')[0].replace('_', ' ').strip()
    return name.lower().strip()

# test on 100 images
dataset_path = '/kaggle/input/datasets/arkaloveskaggle/prescription-preprocessed/preprocessed'
all_images = [f for f in os.listdir(dataset_path) if f.endswith(('.jpg', '.jpeg', '.png'))]
test_images = all_images[-100:]

actuals = []
predictions = []
correct = 0

for filename in tqdm(test_images):
    image_path = os.path.join(dataset_path, filename)
    actual = get_actual_label(filename)
    predicted = predict(image_path)
    
    actuals.append(actual)
    predictions.append(predicted)
    
    if actual in predicted or predicted in actual:
        correct += 1

accuracy = (correct / len(test_images)) * 100
print(f"\nAccuracy : {accuracy:.2f}%")
print(f"CER      : {cer(actuals, predictions):.4f}")
print(f"WER      : {wer(actuals, predictions):.4f}")